## Turn-mode compaction (alternative loader)

Run this cell **instead of** the cell above to load the model with
turn-based eviction. Restart the kernel first if you've already loaded
the block-FIFO LLM (only one engine fits per GPU at this util).

- `compaction_max_turns=4`: trigger when 4 live completed turns sit on top
  of the system prompt
- `compaction_eviction_turn_stride=2`: drop the 2 oldest completed turns
  per fire (block-aligned, inward snap)
- `compaction_turn_end_token_id=None`: auto-detect Qwen3 `<|im_end|>`
  (token id 151645)
- `compaction_window_size=1024`: still required (turn-mode reuses the
  window guard as a hard floor)
- `compaction_stride=256`: ignored in turn-mode but config requires it
  > 0 and block-aligned


In [1]:
# === Compaction config knobs (read by chat() helper below) ===
PAD = True           # if True, pad each message so <|im_end|> lands on a block boundary
BLOCK_SIZE = 16       # must match LLM(block_size=...) below
PAD_FILLER_ID = None  # None = auto (uses tokenizer.pad_token_id, falls back to a space)

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["VLLM_COMPACTION_DEBUG_TOKENS"] = "1"

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen3-4B-Instruct-2507",
    max_model_len=8192,
    enforce_eager=True,
    gpu_memory_utilization=0.85,
    enable_prefix_caching=False,
    block_size=BLOCK_SIZE,
    # Window/stride are still required by the validator; in turn-mode
    # the trigger is `live_completed_turns >= compaction_max_turns`,
    # not the window. Stride is unused (turn-mode plans its own range).
    compaction_window_size=1024,
    compaction_stride=256,
    compaction_protected_prefix_tokens=0,  # turn-mode protects via turn_end_positions[0]
    compaction_max_turns=4,
    compaction_eviction_turn_stride=2,
    compaction_turn_end_token_id=None,  # auto-detect <|im_end|> from tokenizer
    async_scheduling=False,
)
tokenizer = llm.get_tokenizer()
print(f"loaded (turn-mode, PAD={PAD})")


/home/toolkit/kv-eviction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
Enabled int64-addressing Triton patch for vLLM DeepGEMM ep_scatter.


INFO 04-13 04:40:11 [utils.py:233] non-default args: {'max_model_len': 8192, 'block_size': 16, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'async_scheduling': False, 'compaction_window_size': 1024, 'compaction_stride': 256, 'compaction_max_turns': 4, 'compaction_eviction_turn_stride': 2, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
WARNING 04-13 04:40:11 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_COMPACTION_DEBUG_TOKENS
INFO 04-13 04:40:12 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-13 04:40:12 [model.py:1678] Using max model len 8192
INFO 04-13 04:40:12 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-13 04:40:12 [vllm.py:790] Asynchronous scheduling is disabled.
WARNING 04-13 04:40:12 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-13 04:4

(EngineCore pid=86911) `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
(EngineCore pid=86911) Enabled int64-addressing Triton patch for vLLM DeepGEMM ep_scatter.


(EngineCore pid=86911) INFO 04-13 04:40:26 [core.py:105] Initializing a V1 LLM engine (v0.1.dev15360+ga08f29d9c.d20260412) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=86911) INFO 04-13 04:40:31 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/3)
(EngineCore pid=86911) INFO 04-13 04:40:31 [weight_utils.py:825] Prefetching checkpoint files: 30% (3/3)
(EngineCore pid=86911) INFO 04-13 04:40:31 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.77s


Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.69it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.49it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.32it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.05it/s]
(EngineCore pid=86911) 


(EngineCore pid=86911) INFO 04-13 04:40:31 [default_loader.py:384] Loading weights took 1.47 seconds
(EngineCore pid=86911) INFO 04-13 04:40:32 [gpu_model_runner.py:4862] Model loading took 7.61 GiB memory and 4.178851 seconds
(EngineCore pid=86911) INFO 04-13 04:40:33 [gpu_worker.py:436] Available KV cache memory: 57.18 GiB
(EngineCore pid=86911) INFO 04-13 04:40:33 [kv_cache_utils.py:1319] GPU KV cache size: 416,368 tokens
(EngineCore pid=86911) INFO 04-13 04:40:33 [kv_cache_utils.py:1324] Maximum concurrency for 8,192 tokens per request: 50.83x


(EngineCore pid=86911) 2026-04-13 04:40:33,822 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=86911) 2026-04-13 04:40:33,846 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=86911) INFO 04-13 04:40:34 [core.py:283] init engine (profile, create kv cache, warmup model) took 1.68 seconds
(EngineCore pid=86911) WARNING 04-13 04:40:35 [scheduler.py:297] [COMPACT] enabled window=1024 stride=256 protected_prefix=0 max_turns=4 turn_stride=2 turn_end_id=None
(EngineCore pid=86911) INFO 04-13 04:40:35 [vllm.py:790] Asynchronous scheduling is disabled.
(EngineCore pid=86911) WARNING 04-13 04:40:35 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=86911) WARNING 04-13 04:40:35 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=86911) INFO 04-13 04:40:35 [vllm.py:1025] Cudagraph is disabled under eager mode
(EngineCore pid=86911) INFO 04-13 04:40:35 [compilation.py:290] Enabled custom fusions: norm_quant, act_quant
loaded (turn-mo

## Chat helper + tools

Defines `TOOLS` (single `take_action` function) and a `chat()` wrapper
that applies the chat template, calls `llm.chat()`, prints any compaction
events fired during the request, and returns `(text, RequestOutput)`.


In [2]:
from vllm import SamplingParams

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "take_action",
            "description": "Take an action in the BabyAI environment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "action": {
                        "type": "string",
                        "description": "Action to take (e.g. 'forward', 'left', 'right', 'pickup', 'drop', 'toggle', 'done').",
                    }
                },
                "required": ["action"],
            },
        },
    }
]


def _filler_token_id():
    """Pick a filler token id. Override -> tokenizer.pad_token_id ->
    encode a single space. Always returns int."""
    if PAD_FILLER_ID is not None:
        return int(PAD_FILLER_ID)
    pad = getattr(tokenizer, "pad_token_id", None)
    if pad is not None:
        return int(pad)
    enc = tokenizer.encode(" ", add_special_tokens=False)
    if not enc:
        # Last resort: <|endoftext|> if it exists
        eot = tokenizer.convert_tokens_to_ids("<|endoftext|>")
        assert isinstance(eot, int) and eot >= 0, "no usable filler token"
        return eot
    return int(enc[-1])


def _render_padded(messages, tools, im_end_id, block_size, filler_id):
    """Render conversation -> token ids -> insert filler tokens before each
    <|im_end|> so it lands at the LAST slot of a block. The position right
    AFTER <|im_end|> is then a multiple of block_size, matching what the
    turn-mode scheduler tracks in turn_end_positions."""
    # Always go via string + encode to dodge tokenize=True quirks with
    # tools (some chat templates return the rendered string instead of
    # token ids when tools are present).
    rendered = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, tokenize=False
    )
    raw = tokenizer.encode(rendered, add_special_tokens=False)
    # Defensive: ensure all ints
    bad = [(i, type(t).__name__) for i, t in enumerate(raw) if not isinstance(t, int)]
    if bad:
        raise TypeError(f"non-int tokens in encode output: first 5={bad[:5]}")

    out, pads = [], []
    for tok in raw:
        if tok == im_end_id:
            cur = len(out) % block_size
            target = block_size - 1
            n = (target - cur) % block_size
            if n:
                out.extend([filler_id] * n)
            pads.append(n)
        out.append(tok)
    return raw, out, pads


def _decode_evicted(event):
    ids = getattr(event, "evicted_token_ids", None)
    if not ids:
        return None
    try:
        return tokenizer.decode(ids, skip_special_tokens=False)
    except Exception as e:
        return f"<decode-failed: {e}>"


def chat(messages, tools=None, max_tokens=512, temperature=1.0, seed=0,
         show_evicted=True, show_pad_summary=True):
    sp = SamplingParams(max_tokens=max_tokens, temperature=temperature, seed=seed)

    if PAD:
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        assert isinstance(im_end_id, int) and im_end_id >= 0, \
            f"<|im_end|> not in tokenizer (got {im_end_id!r})"
        filler_id = _filler_token_id()
        raw, padded, pads = _render_padded(
            messages, tools, im_end_id, BLOCK_SIZE, filler_id
        )
        if show_pad_summary:
            print(f"  [pad] raw={len(raw)} -> padded={len(padded)} "
                  f"(+{len(padded)-len(raw)}); per-im_end pads={pads} "
                  f"(filler_id={filler_id})")
        outs = llm.generate(
            prompts=[{"prompt_token_ids": padded}],
            sampling_params=sp,
            use_tqdm=False,
        )
    else:
        outs = llm.chat(messages=messages, tools=tools,
                        sampling_params=sp, use_tqdm=False)

    out = outs[0]
    text = out.outputs[0].text

    events = getattr(out, "compaction_events", None) or []
    if events:
        print(f"  [compaction] {len(events)} event(s):")
        for i, ev in enumerate(events):
            line = (
                f"    #{i} evicted={ev.tokens_evicted} "
                f"offset_after={ev.position_offset_after} "
                f"prompt_tokens={getattr(ev, 'num_prompt_tokens', '?')} "
                f"last_turn={getattr(ev, 'last_turn_evicted', '?')} "
                f"turns_evicted_after={getattr(ev, 'num_turns_evicted_after', '?')}"
            )
            print(line)
            if show_evicted:
                decoded = _decode_evicted(ev)
                if decoded is not None:
                    print(f"      evicted-text (full, {len(decoded)} chars):")
                    print("      " + "-" * 60)
                    for ln in decoded.splitlines() or [decoded]:
                        print(f"      | {ln}")
                    print("      " + "-" * 60)
    return text, out


## BALROG eval loop

Reproduces in-the-wild multi-turn compaction degeneration by driving the same `llm` instance through a BabyAI episode.

In [3]:
# BALROG BabyAI eval loop — reuse the existing `llm` and `TOOLS` to reproduce
# in-the-wild multi-turn degeneration. Simple path: treat env responses as
# user messages so we don't need tool-call wire IDs on the replay side.

from balrog.environments import make_env
from omegaconf import OmegaConf
import json as _json
import re as _re

_balrog_cfg = OmegaConf.load("/tmp/balrog/balrog/config/config.yaml")

def reset_env(task="BabyAI-MixedTrainLocal-v0/goto", seed=0):
    env = make_env("babyai", task, _balrog_cfg)
    obs, _ = env.reset(seed=seed)
    return env, obs

def obs_text(obs):
    if isinstance(obs, dict) and "text" in obs and "long_term_context" in obs["text"]:
        return obs["text"]["long_term_context"]
    return str(obs)

_TOOL_CALL_RE = _re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", _re.DOTALL)

def parse_action(text):
    m = _TOOL_CALL_RE.search(text)
    if not m:
        return None
    try:
        blob = _json.loads(m.group(1))
        return blob.get("arguments", {}).get("action")
    except Exception:
        return None


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
# Run a BabyAI episode and dump full per-turn output (assistant text,
# parsed action, env reward, compaction events with full evicted text)
# to out_turn.txt for offline inspection.
#
# IMPORTANT: this loop calls the chat() helper from the cell above, which
# honors the PAD knob set in the loader cell. To get exact-<|im_end|>
# eviction, set PAD=True in the loader cell BEFORE creating the LLM.
import os as _os

OUT_PATH = "/home/toolkit/kv-eviction/experiments/debug_balrog/out_turn.txt"
MAX_TURNS = 10

env, obs = reset_env(task="BabyAI-MixedTrainLocal-v0/pickup", seed=0)
system_prompt = env.get_instruction_prompt(instructions=obs["mission"])

conv = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": obs_text(obs)},
]


def _format_event(i, ev):
    lines = []
    lines.append(
        f"  event #{i}: evicted={ev.tokens_evicted} "
        f"offset_after={ev.position_offset_after} "
        f"prompt_tokens={getattr(ev, 'num_prompt_tokens', '?')} "
        f"last_turn={getattr(ev, 'last_turn_evicted', '?')} "
        f"turns_evicted_after={getattr(ev, 'num_turns_evicted_after', '?')}"
    )
    ids = getattr(ev, "evicted_token_ids", None)
    if ids:
        try:
            decoded = tokenizer.decode(ids, skip_special_tokens=False)
        except Exception as e:
            decoded = f"<decode-failed: {e}>"
        lines.append(f"  evicted-text ({len(decoded)} chars):")
        lines.append("  " + "-" * 70)
        for ln in (decoded.splitlines() or [decoded]):
            lines.append(f"  | {ln}")
        lines.append("  " + "-" * 70)
    return "\n".join(lines)


with open(OUT_PATH, "w") as f:
    f.write(f"# BALROG turn-mode compaction trace (PAD={PAD})\n")
    f.write(f"# task: BabyAI-MixedTrainLocal-v0/pickup, seed=0, max_turns={MAX_TURNS}\n")
    f.write("=" * 80 + "\n")
    f.write("SYSTEM PROMPT\n")
    f.write("=" * 80 + "\n")
    f.write(system_prompt + "\n\n")
    f.write("=" * 80 + "\n")
    f.write("INITIAL USER (obs)\n")
    f.write("=" * 80 + "\n")
    f.write(obs_text(obs) + "\n\n")
    f.flush()

    for turn in range(MAX_TURNS):
        print(f"turn {turn}...", end=" ", flush=True)

        # Route through chat() so PAD is honored.
        text, out = chat(
            conv, tools=TOOLS, max_tokens=2000, temperature=1.0,
            seed=turn, show_evicted=False, show_pad_summary=True,
        )
        events = getattr(out, "compaction_events", None) or []
        action = parse_action(text)

        f.write("=" * 80 + "\n")
        f.write(f"TURN {turn}\n")
        f.write("=" * 80 + "\n")
        f.write(f"prompt_tokens(scheduled)={len(out.prompt_token_ids) if out.prompt_token_ids else '?'} "
                f"output_tokens={len(out.outputs[0].token_ids)} "
                f"events={len(events)}\n\n")

        if events:
            f.write(f"COMPACTION EVENTS ({len(events)}):\n")
            for i, ev in enumerate(events):
                f.write(_format_event(i, ev) + "\n")
            f.write("\n")

        f.write("ASSISTANT:\n")
        f.write(text + "\n\n")
        f.write(f"PARSED ACTION: {action!r}\n\n")

        conv.append({"role": "assistant", "content": text})

        if action is None:
            err = "Error: No valid tool_call found. Emit a <tool_call> with take_action."
            f.write("USER (env error):\n")
            f.write(err + "\n\n")
            f.flush()
            conv.append({"role": "user", "content": err})
            print("(no action)")
            continue

        try:
            valid = env.check_action_validity(action)
            obs, reward, term, trunc, info = env.step(valid)
        except Exception as e:
            err = f"Error stepping env: {e}"
            f.write("USER (env error):\n")
            f.write(err + "\n\n")
            f.flush()
            conv.append({"role": "user", "content": err})
            print(f"(step error: {e})")
            continue

        f.write(f"REWARD={reward} done={term or trunc}\n\n")
        next_obs_txt = obs_text(obs)
        f.write("USER (obs):\n")
        f.write(next_obs_txt + "\n\n")
        f.flush()

        conv.append({"role": "user", "content": next_obs_txt})
        print(f"action={action!r} reward={reward}")
        if term or trunc:
            f.write("=" * 80 + "\n")
            f.write("EPISODE FINISHED\n")
            f.write("=" * 80 + "\n")
            print("episode finished")
            break

print(f"\nfull trace written to {OUT_PATH} (PAD={PAD})")
print(f"size: {_os.path.getsize(OUT_PATH)} bytes")


turn 0...   [pad] raw=368 -> padded=372 (+4); per-im_end pads=[3, 1] (filler_id=151643)
(EngineCore pid=86911) WARNING 04-13 04:40:36 [scheduler.py:1069] [COMPACT] turn mode: using eos_token_id=151645 as message-end marker (set --compaction-turn-end-token-id explicitly to override)
action='forward' reward=0
turn 1...   [pad] raw=519 -> padded=532 (+13); per-im_end pads=[3, 1, 8, 1] (filler_id=151643)
action='forward' reward=0
turn 2...   [pad] raw=701 -> padded=724 (+23); per-im_end pads=[3, 1, 8, 1, 4, 6] (filler_id=151643)
action='forward' reward=0
turn 3...   [pad] raw=823 -> padded=868 (+45); per-im_end pads=[3, 1, 8, 1, 4, 6, 8, 14] (filler_id=151643)
action='forward' reward=0
turn 4...   [pad] raw=930 -> padded=996 (+66); per-im_end pads=[3, 1, 8, 1, 4, 6, 8, 14, 7, 14] (filler_id=151643)
(EngineCore pid=86911) WARNING 04-13 04:40:40 [scheduler.py:1280] [COMPACT] req=4-b0da2d effective_prompt=336 num_prompt=996 evict=[336,688) total=352 generated=1 events=1 turn_mode=True last_tu